# BOI + P96-clip + 3FFT (z^2, z^4, |z|^2) consolidated vector PT generation

`boi_p96_fft3` fork of `boi_p96_fft2`: adds ONE extra FFT (`FFT{|z|^2}`, the envelope
power of the already-clipped `z`) used only as a cross-check/arbitration branch for
`rs_estimate_prominence` (see `rs_prominence_fftz2_correction`), and 8 features for the LR router, which is now `lr8_32` (32-dim, supersedes `lr8_24`) instead of `lr8_24` (see `az2_lr_features` / `LR8_32_FEATURES`). Everything else (BOI/clip/z^4/CFO) is unchanged.

Single data source: `CSPB.ML_dataset` (2018 / 2022 / 2018R2). For each signal this
notebook computes the established BOI+CFO-shift pipeline, then the validated P96-clip
+ z^4 two-FFT front end (plus the new |z|^2 cross-check FFT), and saves ONE consolidated
payload per sample containing:

- `x_processed` (unclipped BOI output, complex64) and `z` (P96-clipped, phase-preserved,
  complex64), plus the clip threshold `clip_thr` itself.
- `power`: `P_x_processed`, `P_z_total`, `p_boi_inband_signal`, `p_boi_inband_signal_z`.
- The z^4 log-magnitude spectrum itself is NOT saved (it's large and purely an
  intermediate); only the Rs/CFO estimates computed from it are kept.
- Rs: `rs_estimate_prominence` (blind circular-prominence full-grid search on z^4,
  verified 98.24% on the full 2018/2022 QPSK-like corpus) with its `rs_score_prominence`,
  AND the independent `rs_estimate_abs_x2_rule` (trust gate -> x4-triplet -> ranked
  fallback, ~95% on 2018R2) with its `rs_abs_x2_rule_source` tag -- computed directly
  from `x_processed`, no intermediate feature-vector scheme -- AND (NEW in boi_p96_fft3)
  `rs_estimate_prominence_corrected`: an `FFT{|z|^2}` cross-check arbitration on top of
  `rs_estimate_prominence` (the 3rd FFT). It only overrides `rs_estimate_prominence` when
  the |z|^2 spectrum's top1/top2 dB gap is confidently trusted (reusing
  `RS20_TRUST_ABS2_DB_GAP_MAX` verbatim, not a threshold newly fit to this task) AND its
  candidate disagrees with `rs_estimate_prominence`; otherwise it is an identity no-op.
  Validated on the full 167,816-sample QPSK-like corpus with an honest group-split
  train/val/test + fully-external (2018R2) breakdown: 0 regressions on ANY split (a
  sample `rs_estimate_prominence` already got right is NEVER changed), 936 samples
  recovered, raising accuracy 97.175%->97.965% (2018), 99.309%->99.429% (2022),
  97.355%->98.120% (2018R2). See `rs_prominence_fftz2_correction`'s docstring in
  `helper_function/pre_boi_p96_fft2_generator.py` for the full validation detail.
- CFO: `cfo_estimate_argmax` (original unrestricted-argmax, kept for comparison) AND
  `cfo_estimate_band_interp` (band-limited + 3-point parabolic interpolation, closes the
  downstream classification gap to ground-truth CFO to within +/-0.4pp).
- `t0_estimate` (= 1/rs_estimate_prominence, the UNCORRECTED estimate, unchanged).
- `lr8_32`: the 32-dim float32 feature vector for the LR8_32 5-class coarse router
  (BPSK/MSK/DQPSK/8PSK/QPSK-like) -- REPLACES the old 24-dim `lr8_24` (removed). 24 base
  features from the top-8 separated peaks of the Hann `FFT{z^2}` and `FFT{z^4}` spectra
  (same functions as the validated `FPGA_pytorch_sim_2FFT_z-test` generator) PLUS 8
  features from the top-16 peaks / full spectrum of `FFT{|z|^2}` (the 3rd FFT, shares the
  spectrum already computed for the Rs correction above -- no 4th FFT), picked by a
  5-fold-CV forward search over a 41-candidate pool: 0 regressions on any of the 4
  held-out metrics (2018train20/2022train20 same-domain + external), every metric
  improved +0.026pp to +0.050pp. Feature names are stored once per batch in
  `pack["lr8_32_feature_names"]`. The spectra and peak lists themselves are NOT saved.
- `truth` (every raw truth.txt column, renamed snake_case, plus rs_truth/cfo_truth/
  cfo_truth_shifted/snr_truth_db aliases) and `preprocess` (BOI window/shift details).

The old 445/55/28/20/7-feature vector scheme is NOT computed or saved here -- it added
compute cost and file size for information this notebook doesn't need.

ALL signal types are processed (the LR router needs every class). The Rs/CFO estimators
are QPSK-like-specific and give meaningless (but finite) values for bpsk/8psk/dqpsk/msk;
the QPSK-like filter is applied downstream by the CC-matrix generation notebooks.
2018 negative-SNR samples are detected from the truth row alone (no BOI/clip/FFT is ever
run for them) and stored with `status=False`, `error="SKIP_NEGATIVE_TRUTH_SNR"`; processing
failures likewise get `status=False` with the exception recorded in `error`. Every signal in
the batch's index range is included regardless of outcome, so `len(pack["samples"])` always
equals the batch's signal count -- nothing is silently dropped.

Output: `generated_boi_p96_fft2_pt/{{dataset}}_batch_{{NN}}_boi_p96_fft2_full.pt` (filenames and
folder name are UNCHANGED from `boi_p96_fft2` -- only this project's own copy of the
generator/notebooks differs; `pack["version"]` is bumped to `"boi_p96_fft2_full_v4"` to
mark the schema change, lr8_24 -> lr8_32). All three datasets' files share this one subfolder.


In [ ]:
from pathlib import Path
import sys


# Run with the Jupyter working directory set to this notebook's folder.
PROJECT_DIR = Path(".")
if not (PROJECT_DIR / "helper_function" / "__init__.py").is_file():
    raise FileNotFoundError("Run this notebook from the project directory containing helper_function.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from helper_function.pre_boi_p96_fft2_generator import (
    generate_boi_p96_fft2_pt_files,
    inspect_boi_p96_fft2_pt,
    GENERATION_METHOD_VERSION,
)

BASE = PROJECT_DIR
DATASET_ROOT = BASE / "../../CSPB.ML_dataset"
OUTPUT_ROOT = BASE / "generated_boi_p96_fft2_pt"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("generation_method_version:", GENERATION_METHOD_VERSION)


## Generation Config

Output goes to `generated_boi_p96_fft2_pt/` under this notebook's directory (created if
missing); it produces `2018_batch_NN_boi_p96_fft2_full.pt` files for
`DATASET_SELECT="2018"`.


In [ ]:
CFG = {
    "ROOT_MAP": {
        # Dataset directories relative to the project working directory.
        "2022": str(DATASET_ROOT / "CSPB_ML_2022_Data"),
        "2018": str(DATASET_ROOT / "CSPB_ML_2018_Data"),
        "2018R2": str(DATASET_ROOT / "CSPB_ML_2018R2_Data"),
    },
    "LABEL_MAP": {
        "2022": "CSPB_ML_2022_Signal_Truth_Labels.txt",
        "2018": "CSPB_ML_2018_Signal_Truth_Labels.txt",
        "2018R2": "CSPB_ML_2018R2_Signal_Truth_Labels.txt",
    },
    "N_PER_BATCH": 4000,
    "N_BATCHES": 28,
    "N_TOTAL": 4000 * 28,
    "SAMPLES_PER_SIG": 32768,
    "FS_IN": 1.0,
    "EPS": 1e-12,
    "OUT_ROOT": str(OUTPUT_ROOT),
    "SKIP_EXISTING": False,
    "DATASET_SELECT": "2018",
    "START_PT": 1,
    "END_PT": 28,
    "DELTA_F0": 0.002,
    "THRESHOLD_DB": 5.0,
    "BOUNDARY_THRESHOLD_DB": 1.5,
    "BOI_SELECT_MODE": "strongest",
    "MERGE_GAP_W_FACTOR": 2.0,
    "MAX_EXPAND_W_FACTOR": 8.0,
    "BOI_GUARD_BINS": 2,
    "ADAPTIVE_FILTER_EXPAND": True,
    "FILTER_EXPAND_RATIO": 0.20,
    "FILTER_EXPAND_RATIO_LOW_SNR": 0.30,
    "FILTER_EXPAND_RATIO_MID_SNR": 0.15,
    "FILTER_EXPAND_RATIO_HIGH_SNR": 0.05,
    "SNR_LOW_TH_DB": 4.0,
    "SNR_HIGH_TH_DB": 8.0,
    "MAX_EXPANDED_FILTER_BW_FRAC": 0.95,
    "FFT_NFFT": 32768,
    "FFT_WINDOW": "hann",
    "FFT_TOP_K": 20,
    "FFT_PEAK_MIN_SEP_BINS": 8,
    "GUIDE_KMAX_INTEGER_RS": 5,
    "GUIDE_KMAX_HALF_RS": 5,
    "GUIDE_SEARCH_BINS": 3,
}

print("Will generate to:", CFG["OUT_ROOT"])
print("Dataset select:", CFG["DATASET_SELECT"], "batches:", CFG["START_PT"], "to", CFG["END_PT"])


## Run Generation

Performs the full `2018` generation from `CSPB.ML_dataset`. No CSV/cache
files are written; every sample (success, negative-SNR skip, or error) is kept inside
its batch's `.pt` file with an explicit `status` flag.


In [ ]:
generate_boi_p96_fft2_pt_files(CFG)


## Inspect One Output File

Run after generation.


In [4]:
example_pt = OUTPUT_ROOT / "2018_batch_01_boi_p96_fft2_full.pt"
inspect_boi_p96_fft2_pt(example_pt)


{'version': 'boi_p96_fft2_full_v4',
 'dataset_name': '2018',
 'batch_id': 1,
 'num_samples': 4000,
 'num_ok': 3992,
 'num_negative_snr_skipped': 8,
 'num_error': 0,
 'lr8_32_feature_names': ['x2_top5_f',
  'x2_top8_f',
  'x2_top5_minus_top6_db',
  'x2_energy_ratio_top4_of_8',
  'x2_sym8_score_min',
  'x2_sym8_score_min2',
  'x4_top1_af',
  'x4_top2_af',
  'x4_top3_db',
  'x4_top6_af',
  'x4_top7_f',
  'x4_top8_f',
  'x4_top8_af',
  'x4_top1_minus_top2_db',
  'x4_top2_minus_top3_db',
  'x4_top6_minus_top7_db',
  'x4_count_gt_m20_d8',
  'x4_sym8_score_min',
  'x4_sym8_score_min2',
  'x2_top3_db',
  'x4_top7_minus_top8_db',
  'x2_top1_minus_top2_db',
  'x2_top7_f',
  'x4_top2_f',
  'az2_peak16_energy_ratio',
  'az2_p1_db',
  'az2_p6_db',
  'az2_p3_minus_p5_db',
  'az2_p2_minus_p3_db',
  'az2_p3af_over_p1af',
  'az2_p5_minus_p6_db',
  'az2_count_gt_m10_d8'],
 'example_ok_sample_keys': ['cfo_estimate_argmax',
  'cfo_estimate_band_interp',
  'clip_thr',
  'error',
  'lr8_32',
  'meta',
  'po